In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score

In [2]:
Data = pd.read_csv('synthetic_mof_synthesis.csv')
Data.head(2)

,sample_id,mof_family,metal_salt,linker,solvent,modulator,modulator_equiv,metal_linker_ratio,concentration_M,temperature_C,time_h,batch,relative_crystallinity,phase_outcome,yield_pct,BET_m2_g
0,M0000,MIL-53(Al),Al(NO3)3,terephthalic acid,H2O,none,0,0.84,0.104,210.7,35.30,6,0.673,phase-pure MOF,80.0,609.0
1,M0001,UiO-66,ZrCl4,terephthalic acid,DMF,none,0,0.92,0.091,108.1,8.49,8,0.751,phase-pure MOF,68.2,818.0


In [3]:
Data.isnull().sum()

sample_id                  0
mof_family                 0
metal_salt                 0
linker                     0
solvent                    0
modulator                  0
modulator_equiv            0
metal_linker_ratio         0
concentration_M            0
temperature_C              0
time_h                     0
batch                      0
relative_crystallinity     0
phase_outcome              0
yield_pct                 40
BET_m2_g                  50
dtype: int64

In [4]:
Data[['metal_linker_ratio',	'concentration_M',	'temperature_C',	'phase_outcome']]

,metal_linker_ratio,concentration_M,temperature_C,phase_outcome
0,0.84,0.104,210.700000,phase-pure MOF
1,0.92,0.091,108.100000,phase-pure MOF
2,6.90,0.065,67.600000,amorphous/other
3,1.27,0.159,212.500000,phase-pure MOF
4,1.75,0.063,163.300000,mixed phase
...,...,...,...,...
655,1.23,0.214,94.240404,amorphous/other
656,1.15,0.179,219.754442,phase-pure MOF
657,8.64,0.056,68.186884,amorphous/other
658,0.79,0.058,104.976443,phase-pure MOF


In [5]:
X = Data.iloc[:, [7, 8, 9]]
X.head()

,metal_linker_ratio,concentration_M,temperature_C
0,0.84,0.104,210.7
1,0.92,0.091,108.1
2,6.90,0.065,67.6
3,1.27,0.159,212.5
4,1.75,0.063,163.3


In [6]:
Y = (Data.iloc[:, 12]>0.6).astype(int)
Y

0      1
1      1
2      0
3      1
4      0
      ..
655    0
656    1
657    0
658    1
659    1
Name: relative_crystallinity, Length: 660, dtype: int64

In [30]:
Y.info()

<class 'pandas.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 1 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   relative_crystallinity  660 non-null    float64
dtypes: float64(1)
memory usage: 5.3 KB


In [7]:
np.mean(cross_val_score(LogisticRegression(), X, Y, scoring ='accuracy', cv=20))

np.float64(0.6227272727272727)

## Applying Feature Construction

In [22]:
X['metal_linker_content'] = np.log(X['metal_linker_ratio'] + X['concentration_M']) 
X['metal_linker_content']

0     -0.057629
1      0.010940
2      1.940898
3      0.356975
4      0.594983
         ...   
655    0.367417
656    0.284427
657    2.162863
658   -0.164875
659    0.004988
Name: metal_linker_content, Length: 660, dtype: float64

In [23]:
np.round(X['metal_linker_content'])

0     -0.0
1      0.0
2      2.0
3      0.0
4      1.0
      ... 
655    0.0
656    0.0
657    2.0
658   -0.0
659    0.0
Name: metal_linker_content, Length: 660, dtype: float64

In [24]:
X

,metal_linker_ratio,concentration_M,temperature_C,metal_linker_content
0,0.84,0.104,210.700000,-0.057629
1,0.92,0.091,108.100000,0.010940
2,6.90,0.065,67.600000,1.940898
3,1.27,0.159,212.500000,0.356975
4,1.75,0.063,163.300000,0.594983
...,...,...,...,...
655,1.23,0.214,94.240404,0.367417
656,1.15,0.179,219.754442,0.284427
657,8.64,0.056,68.186884,2.162863
658,0.79,0.058,104.976443,-0.164875


In [25]:
np.mean(cross_val_score(LogisticRegression(), X, Y, scoring ='accuracy', cv=20))

np.float64(0.6242424242424243)

In [26]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3)

X_train

,metal_linker_ratio,concentration_M,temperature_C,metal_linker_content
99,0.80,0.091,64.800000,-0.115411
653,1.59,0.229,64.526791,0.598287
385,5.88,0.047,53.200000,1.779518
19,7.27,0.154,74.900000,2.004718
24,1.94,0.259,86.000000,0.788003
...,...,...,...,...
458,0.85,0.050,85.400000,-0.105361
584,1.72,0.049,179.500000,0.570414
622,0.85,0.282,90.767864,0.123986
258,7.57,0.083,23.800000,2.035098


In [27]:
clf = LogisticRegression()

clf.fit(X_train, Y_train)

Y_pred = clf.predict(X_test)

accuracy_score(Y_pred, Y_test)

0.6262626262626263

### Feature splitting

In [39]:
df = pd.read_csv('messy_data.csv')
df.head()

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,ASTHMA,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,M,16,DIABETES,Jul 13 2023,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,F,92.0,asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com


In [40]:

df2 = df['weight'].to_frame()
df2

,weight
0,105.9 kg
1,105.3
2,65.4 kg
3,60.5
4,68.7 kg
...,...
424,88.1 kg
425,74.2 kg
426,56.3
427,55.9


In [48]:
df2['units'] = df2['weight'].str.split(' ', expand = True)[1]
df2['units']

0       kg
1      NaN
2       kg
3      NaN
4       kg
      ... 
424     kg
425     kg
426    NaN
427    NaN
428    NaN
Name: units, Length: 429, dtype: str

In [58]:
df2['weight'] = pd.to_numeric(df2['weight'].str.split(' ', expand = True)[0], errors = 'coerce')
df2['weight']

0      105.9
1      105.3
2       65.4
3       60.5
4       68.7
       ...  
424     88.1
425     74.2
426     56.3
427     55.9
428     66.8
Name: weight, Length: 429, dtype: float64

In [59]:
df2[['weight', 'units']]

,weight,units
0,105.9,kg
1,105.3,NaN
2,65.4,kg
3,60.5,NaN
4,68.7,kg
...,...,...
424,88.1,kg
425,74.2,kg
426,56.3,NaN
427,55.9,NaN
